# Kerala IDSP Hybrid Chatbot

## Notebook 00: Project Definition and Data Audit

### Objective

This notebook checks:

- whether the PDF files are available;
- whether each report is daily or weekly;
- the visible report date;
- the number of pages;
- whether the PDF contains readable text;
- which pages contain tables.

No disease information will be stored or embedded yet.

In [1]:
%pip install -q pandas pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from datetime import datetime
import re

import pandas as pd
import pdfplumber

In [3]:
CURRENT_FOLDER = Path.cwd().resolve()

if (CURRENT_FOLDER / "data").exists():
    PROJECT_ROOT = CURRENT_FOLDER

elif (CURRENT_FOLDER.parent / "data").exists():
    PROJECT_ROOT = CURRENT_FOLDER.parent

else:
    raise FileNotFoundError(
        "Project folder not found. Make sure this notebook is "
        "inside the notebooks folder."
    )

DAILY_PDF_FOLDER = PROJECT_ROOT / "data" / "raw" / "daily"
WEEKLY_PDF_FOLDER = PROJECT_ROOT / "data" / "raw" / "weekly"

print("Project root:", PROJECT_ROOT)
print("Daily PDF folder:", DAILY_PDF_FOLDER)
print("Weekly PDF folder:", WEEKLY_PDF_FOLDER)

Project root: C:\Users\vinee\Downloads\rag_chatbot_kerala
Daily PDF folder: C:\Users\vinee\Downloads\rag_chatbot_kerala\data\raw\daily
Weekly PDF folder: C:\Users\vinee\Downloads\rag_chatbot_kerala\data\raw\weekly


In [4]:
daily_pdfs = sorted(DAILY_PDF_FOLDER.glob("*.pdf"))
weekly_pdfs = sorted(WEEKLY_PDF_FOLDER.glob("*.pdf"))

all_pdfs = daily_pdfs + weekly_pdfs

print("Daily PDFs found:", len(daily_pdfs))
print("Weekly PDFs found:", len(weekly_pdfs))
print("Total PDFs found:", len(all_pdfs))

for pdf_path in all_pdfs:
    print("-", pdf_path.name)

Daily PDFs found: 1
Weekly PDFs found: 0
Total PDFs found: 1
- IDSP-Daily-Report-01.09.2026.pdf


In [5]:
if not all_pdfs:
    raise FileNotFoundError(
        "No PDF was found. Put the report inside:\n"
        "data/raw/daily/"
    )

print("PDF check passed.")

PDF check passed.


In [6]:
def detect_report_type(filename, text):
    combined_text = f"{filename} {text}".lower()

    if "weekly" in combined_text:
        return "weekly"

    if "daily" in combined_text:
        return "daily"

    return "unknown"


def detect_report_date(filename, text):
    combined_text = f"{filename} {text}"

    date_patterns = [
        # Example: September 1, 2026
        (r"\b[A-Za-z]+\s+\d{1,2},\s+\d{4}\b", "%B %d, %Y"),

        # Example: 01-09-26
        (r"\b\d{2}-\d{2}-\d{2}\b", "%d-%m-%y"),

        # Example: 01.09.2026
        (r"\b\d{2}\.\d{2}\.\d{4}\b", "%d.%m.%Y"),
    ]

    for pattern, date_format in date_patterns:
        match = re.search(pattern, combined_text)

        if match:
            try:
                report_date = datetime.strptime(
                    match.group(),
                    date_format
                )

                return report_date.date().isoformat()

            except ValueError:
                continue

    return None

In [7]:
def audit_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        page_count = len(pdf.pages)

        page_texts = []

        for page in pdf.pages:
            page_text = page.extract_text() or ""
            page_texts.append(page_text)

        full_text = "\n".join(page_texts)

        total_characters = len(full_text.strip())

        if total_characters > 100:
            pdf_type = "text-based"
        else:
            pdf_type = "possibly scanned"

        report_type = detect_report_type(
            pdf_path.name,
            full_text
        )

        report_date = detect_report_date(
            pdf_path.name,
            full_text
        )

        file_size_kb = round(
            pdf_path.stat().st_size / 1024,
            2
        )

        return {
            "filename": pdf_path.name,
            "report_date": report_date,
            "report_type": report_type,
            "page_count": page_count,
            "file_size_kb": file_size_kb,
            "text_characters": total_characters,
            "pdf_type": pdf_type,
        }

In [8]:
audit_results = []

for pdf_path in all_pdfs:
    result = audit_pdf(pdf_path)
    audit_results.append(result)

inventory_df = pd.DataFrame(audit_results)

inventory_df

,filename,report_date,report_type,page_count,file_size_kb,text_characters,pdf_type
0,IDSP-Daily-Report-01.09.2026.pdf,2026-09-01,daily,3,355.75,4615,text-based


In [10]:
SAMPLE_PDF = all_pdfs[0]

print("Selected PDF:", SAMPLE_PDF.name)

Selected PDF: IDSP-Daily-Report-01.09.2026.pdf


In [11]:
with pdfplumber.open(SAMPLE_PDF) as pdf:

    for page_number, page in enumerate(pdf.pages, start=1):

        text = page.extract_text() or ""

        print("=" * 60)
        print("PAGE:", page_number)
        print("=" * 60)

        print(text[:500])
        print()

PAGE: 1
NHM/227/2026-X3 IDSP I/370530/2026
State Surveillance Unit
Directorate of Health Services, Kerala
Thiruvananthapuram-695035
Email: idspkerala2020@gmail.com
Communicable Diseases - Daily Report on September 1, 2026
Dr Reetha K P
ADHS(PH) & State Surveillance Officer

PAGE: 2
NHM/227/2026-X3 IDSP I/370530/2026
DISTRICT WISE DAILY REPORTING FORMAT Kerala 01-09-26
.o N Dist. OP Fever IP Sus C C G on D Sus DEN C G on UE D Sus LEP C T o O n D D D A o p C x He A pati C Su C s h D . ole C r C a on D S E A C n o C C E J Imp PV Ind Imp M PF a I la n r d ia Imp Mx Ind D b u r c S n e u lf n I a z
1 TVM 786 13 - 1 - 29 20 1 - 3 - 128 5 - - - - - - - - - - - - - - 1 11
2 KLM 487 1 - - - 18 11 - - 2 - 72 8 3 - - - - - - - - 1 - - - - - -
3 PTA 409 4 - - - 2 3 - - 3 - 48 6 - - - 

PAGE: 3
NHM/227/2026-X3 IDSP I/370530/2026
ANALYSIS OF COMMUNICABLE DISEASES 01-09-26
Daily Present Month Cumulative 2026
Sl.No. Disease
Sus.C Sus.D Con Death Sus.C Sus.D Con Death Sus.C Sus.D Con Death
1 Fever - - 

In [12]:
TABLE_SETTINGS = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "intersection_tolerance": 5,
    "snap_tolerance": 3,
    "join_tolerance": 3,
}

In [13]:
table_results = []

with pdfplumber.open(SAMPLE_PDF) as pdf:

    for page_number, page in enumerate(pdf.pages, start=1):

        tables = page.extract_tables(TABLE_SETTINGS)

        if not tables:
            table_results.append({
                "page": page_number,
                "table_number": None,
                "rows": 0,
                "columns": 0,
            })

        for table_number, table in enumerate(tables, start=1):

            row_count = len(table)

            column_count = max(
                len(row) for row in table
            )

            table_results.append({
                "page": page_number,
                "table_number": table_number,
                "rows": row_count,
                "columns": column_count,
            })

tables_df = pd.DataFrame(table_results)

tables_df

,page,table_number,rows,columns
0,1,NaN,0,0
1,2,1.0,40,31
2,3,1.0,32,15


In [14]:
question_definitions = pd.DataFrame([
    {
        "user_question": "What diseases are reported in Kannur?",
        "meaning": (
            "Diseases with confirmed cases greater than zero "
            "in Kannur in the latest available report."
        )
    },
    {
        "user_question": "Which disease is highest in Kannur?",
        "meaning": (
            "The disease with the largest confirmed count "
            "in Kannur in the latest available report."
        )
    },
    {
        "user_question": "What is the latest disease in Kannur?",
        "meaning": (
            "This is ambiguous. Ask whether the user means "
            "currently reported, newly reported, or highest."
        )
    },
    {
        "user_question": "Did dengue increase?",
        "meaning": (
            "Compare dengue counts between reports of the "
            "same period type: daily with daily or weekly with weekly."
        )
    }
])

question_definitions

,user_question,meaning
0,What diseases are reported in Kannur?,Diseases with confirmed cases greater than zer...
1,Which disease is highest in Kannur?,The disease with the largest confirmed count i...
2,What is the latest disease in Kannur?,This is ambiguous. Ask whether the user means ...
3,Did dengue increase?,Compare dengue counts between reports of the s...


In [15]:
checks = {
    "PDF found": len(all_pdfs) > 0,
    "Report date detected": inventory_df["report_date"].notna().all(),
    "Report type detected": (
        inventory_df["report_type"] != "unknown"
    ).all(),
    "PDF contains readable text": (
        inventory_df["pdf_type"] == "text-based"
    ).all(),
    "Table found on page 2": (
        (tables_df["page"] == 2)
        & (tables_df["rows"] > 0)
    ).any(),
    "Table found on page 3": (
        (tables_df["page"] == 3)
        & (tables_df["rows"] > 0)
    ).any(),
}

for check_name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status}: {check_name}")

if all(checks.values()):
    print("\nNotebook 00 completed successfully.")
else:
    print("\nSome checks failed. Do not start extraction yet.")

PASS: PDF found
PASS: Report date detected
PASS: Report type detected
PASS: PDF contains readable text
PASS: Table found on page 2
PASS: Table found on page 3

Notebook 00 completed successfully.
